In [1]:
'''
Title: Data Wrangling Project
Name: Sandra Lung'ahi
Date: 23rd September, 2025

Description: A project to practice data wrangling concepts using the Netflix dataset.
'''

"\nTitle: Data Wrangling Project\nName: Sandra Lung'ahi\nDate: 23rd September, 2025\n\nDescription: A project to practice data wrangling concepts using the Netflix dataset.\n"

In [2]:
# Import libraries
import pandas as pd
import numpy as np


In [3]:
# Load Dataset
df = pd.read_csv("/kaggle/input/netflix-shows/netflix_titles.csv")
print("Dataset loaded successfully")

Dataset loaded successfully


In [4]:
# Data Discovery

# Print dataset information summary.
print("\n--- Dataset Info ---\n")
df.info()

# Print the shape of the dataset.
print("\nShape (Rows x Columns):", df.shape)

# Print the list of all column names
print("\nColumns:", df.columns.tolist())

# Print the data types of each column.
print("\nData types:\n", df.dtypes)

# Print the number of missing values per column.
print("\nMissing values per column:\n", df.isnull().sum())

# Print the total number of duplicate rows in the dataset.
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Print the first 5 rows.
print("\nSample rows:\n", df.head())



--- Dataset Info ---

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB

Shape (Rows x Columns): (8807, 12)

Columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Data types:
 show_id         object
type  

In [5]:
# Structuring

# Convert 'date_added' to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract duration value & unit.
df[['duration_value', 'duration_unit']] = df['duration'].str.extract(r'(\d+)\s*(\w+)')
df['duration_value'] = pd.to_numeric(df['duration_value'], errors='coerce')

print("\n--- Structured Columns Sample ---\n", df[['duration', 'duration_value', 'duration_unit']].head())


--- Structured Columns Sample ---
     duration  duration_value duration_unit
0     90 min            90.0           min
1  2 Seasons             2.0       Seasons
2   1 Season             1.0        Season
3   1 Season             1.0        Season
4  2 Seasons             2.0       Seasons


In [6]:
# Data Cleaning and Imputation

# Remove duplicates
df = df.drop_duplicates()

# Create director-cast pair column
df['dir_cast'] = df['director'].fillna('') + '---' + df['cast'].fillna('')

# Count frequency of director-cast pairs
pair_counts = df['dir_cast'].value_counts()
valid_pairs = pair_counts[pair_counts >= 3].index

# Build mapping for cast → director.
dir_cast_map = {}
for pair in valid_pairs:
    director, cast = pair.split('---')
    if cast not in dir_cast_map:   # assign first valid mapping
        dir_cast_map[cast] = director

# Impute missing directors using cast
df['director'] = df.apply(
    lambda row: dir_cast_map.get(row['cast'], row['director']), axis=1
)

# Fill remaining missing directors
df['director'] = df['director'].fillna('Not Given')

# Imputation of Country.
dir_country_map = (
    df.dropna(subset=['director', 'country'])
      .drop_duplicates(subset=['director'])
      .set_index('director')['country']
      .to_dict()
)

df['country'] = df.apply(
    lambda row: dir_country_map.get(row['director'], row['country']), axis=1
)

df['country'] = df['country'].fillna('Unknown')

# Cast
df['cast'] = df['cast'].fillna('Not Given')

# Rating
df['rating'] = df['rating'].fillna('Not Rated')

# Date Added 
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])

# Drop helper column
df.drop(columns=['dir_cast'], inplace=True)

# Normalize season labels
df['duration_unit'] = df['duration_unit'].replace({'Seasons': 'Season'})

# Fill missing duration with 'Unknown'
df['duration'] = df['duration'].fillna('Unknown')

# For missing duration_value, also fill defaults
df['duration_value'] = df['duration_value'].fillna(0).astype(int)
df['duration_unit'] = df['duration_unit'].fillna('Unknown')

print("\n--- Missing values after imputation ---\n", df.isnull().sum())

# Print the first 5 rows.
print("\nSample rows:\n", df.head())


--- Missing values after imputation ---
 show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
description       0
duration_value    0
duration_unit     0
dtype: int64

Sample rows:
   show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water        Not Given   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans        Not Given   
4      s5  TV Show           Kota Factory        Not Given   

                                                cast        country  \
0                                          Not Given  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...         Fra

In [7]:
# Error Checks

import datetime as dt

# Find rows where date_added < release_year
invalid_years = df['date_added'].dt.year < df['release_year']
print("Number of inconsistent rows:", invalid_years.sum())

# Inspect some inconsistent rows
print("\n--- Inconsistent Sample ---\n", 
      df.loc[invalid_years, ['title', 'date_added', 'release_year']].head())

# Fix by replacing release_year with date_added year
df.loc[invalid_years, 'release_year'] = df.loc[invalid_years, 'date_added'].dt.year

# Confirm fix
print("\nRemaining inconsistencies:", 
      (df['date_added'].dt.year < df['release_year']).sum())


Number of inconsistent rows: 14

--- Inconsistent Sample ---
                    title date_added  release_year
1551               Hilda 2020-12-14          2021
1696        Polly Pocket 2020-11-15          2021
2920       Love Is Blind 2020-02-13          2021
3168        Fuller House 2019-12-06          2020
3287  Maradona in Mexico 2019-11-13          2020

Remaining inconsistencies: 0


In [8]:
# Validation

# Drop any helper columns used for wrangling
df = df.drop(columns=['dir_cast'], errors='ignore')

# Check column data types
print("\n--- Data Types ---\n", df.dtypes)

# Ensure date_added is datetime
assert np.issubdtype(df['date_added'].dtype, np.datetime64)

# Ensure duration_value is numeric
assert pd.api.types.is_numeric_dtype(df['duration_value'])

# Apply sanity/business logic rules
# Example: remove rows where release_year < 1997 (Netflix launched streaming in 1997)
df = df[df['release_year'] >= 1997]

# Check for missing fields
print("\n--- Missing Values Check ---\n", df.isnull().sum())

# Sample rows for visual inspection
print("\n--- Sample Records ---\n", df.sample(5))

df_reset = df.reset_index(drop=True)



--- Data Types ---
 show_id                   object
type                      object
title                     object
director                  object
cast                      object
country                   object
date_added        datetime64[ns]
release_year               int64
rating                    object
duration                  object
listed_in                 object
description               object
duration_value             int64
duration_unit             object
dtype: object

--- Missing Values Check ---
 show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
description       0
duration_value    0
duration_unit     0
dtype: int64

--- Sample Records ---
      show_id     type                                   title  \
893     s894    Movie           Wave of Cinema: Filosofi Kopi   
4264   s4265  TV Show   

In [9]:
# Publish
# Save cleaned dataset
df_reset.to_csv('/kaggle/working/cleaned_netflix.csv', index=False)
print("Cleaned dataset saved as cleaned_netflix.csv")


Cleaned dataset saved as cleaned_netflix.csv
